# Case study: verified purchases, ratings and helpful votes

Session 5 · Exploratory analysis and statistics recap. Author: course team, licence CC-BY-4.0.

This notebook follows the three practice tasks of the session:

1. **Block 1**: rating distribution, reviews per month, and the redesign of a poor chart.
2. **Block 2**: do verified purchases rate differently? t-test, Mann–Whitney test, confidence interval and effect size; label × verified purchase with chi-square and Cramér's V.
3. **Block 3**: correlation of text length and helpful votes, a check for confounding by review age, and a one-page summary for a product manager.

Cells marked **Exercise** are for you. The theory is in `../theory/01-describing-data.md` to `04-correlation-and-communication.md`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from scipy.stats.contingency import association

# find the repository root (the folder that contains case-study/)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
reviews = pd.read_parquet(ROOT / "case-study" / "data" / "train_sample.parquet")
reviews["n_words"] = reviews["text"].str.split().str.len()
reviews["year"] = reviews["date"].dt.year
OKABE_ITO = {"neg": "#D55E00", "neu": "#999999", "pos": "#0072B2"}
print(reviews.shape)
reviews.head(3)

## Block 1 · Describe the data and design charts

### 1.1 Variable types and summaries

`rating` is ordinal, `helpful_vote` and `n_images` are counts, `verified_purchase` and `label` are nominal, `date` is a date. Summaries should match the type.

In [ ]:
summary = reviews[["rating", "helpful_vote", "n_words", "n_images"]].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).round(2)
summary

In [ ]:
# counts and shares: report both
pd.DataFrame({"n": reviews["rating"].value_counts().sort_index(),
              "share": reviews["rating"].value_counts(normalize=True).sort_index().round(3)})

### 1.2 Rating distribution and number of reviews per month

In [ ]:
counts = reviews["rating"].value_counts().sort_index()
monthly = reviews.set_index("date").resample("MS").size()

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8), layout="constrained")
bars = left.bar(counts.index, counts, color="0.4", width=0.7)
bars[0].set_color("#D55E00")
left.bar_label(bars, labels=[f"{v:.0%}" for v in counts / counts.sum()], padding=2)
left.set(xlabel="star rating", ylabel="reviews", title="Ratings are J-shaped: 62 % five stars, 13 % one star")
left.yaxis.set_major_formatter("{x:,.0f}")
left.spines[["top", "right"]].set_visible(False)

monthly.loc["2010":].plot(ax=right, color="#0072B2")
right.set(xlabel="", ylabel="reviews per month (sample)", title="Review volume grew until 2020")
right.spines[["top", "right"]].set_visible(False)
print(monthly.idxmax().date(), monthly.max())

### 1.3 Improve a poorly designed chart

The chart below has a truncated axis, rainbow colours, no units and a title that says nothing.

**Exercise.** Redesign it with the four critique questions (question, encoding, accuracy, clarity). Decide which measure a product manager cares about (mean rating or share of 1–2 star reviews), start bars at zero, use one colour, and write the finding as the title.

In [ ]:
yearly = reviews.query("year >= 2014").groupby("year")["rating"].agg(
    avg="mean", share_neg=lambda r: (r <= 2).mean())

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(yearly.index, yearly["avg"], color=plt.cm.rainbow(np.linspace(0, 1, len(yearly))))
ax.set_ylim(3.85, 4.22)
ax.set_title("Ratings");

In [ ]:
# Exercise: your improved chart
# fig, ax = plt.subplots(figsize=(6, 3.5), layout="constrained")
# ...

## Block 2 · Do verified purchases rate differently?

### 2.1 Numeric outcome: t-test and Mann–Whitney test

We compare the star rating of verified and unverified purchases with Welch's t-test (means) and the Mann–Whitney test (ranks), and report the difference with a 95 % confidence interval, Cohen's d and the common-language effect size (CLES).

In [ ]:
def compare_groups(df: pd.DataFrame, column: str, group: str = "verified_purchase") -> pd.Series:
    """Welch t-test, Mann-Whitney U, CI and effect sizes for a two-group comparison."""
    a = df.loc[df[group], column]
    b = df.loc[~df[group], column]
    t = stats.ttest_ind(a, b, equal_var=False)
    ci = t.confidence_interval()
    mw = stats.mannwhitneyu(a, b)
    pooled_sd = np.sqrt(((len(a) - 1) * a.var() + (len(b) - 1) * b.var()) / (len(a) + len(b) - 2))
    return pd.Series({
        "n verified": len(a), "n not verified": len(b),
        "mean verified": a.mean(), "mean not verified": b.mean(),
        "difference": a.mean() - b.mean(), "CI low": ci.low, "CI high": ci.high,
        "p (Welch t)": t.pvalue, "p (Mann-Whitney)": mw.pvalue,
        "Cohen's d": (a.mean() - b.mean()) / pooled_sd,
        "CLES": mw.statistic / (len(a) * len(b)),
    })

compare_groups(reviews, "rating").round(4)

**Reading.** The difference is about −0.03 stars, the confidence interval includes zero, and d ≈ −0.02. Verified buyers do not rate differently in any practically relevant way.

What happens with nine times as many reviews? The full training set has 434,373 reviews.

In [ ]:
full = pd.read_parquet(ROOT / "case-study" / "data" / "train.parquet",
                       columns=["rating", "label", "verified_purchase", "date"])
compare_groups(full, "rating").round(4)

With the full data the same tiny difference becomes "significant" (p ≈ 0.02), while d stays at about −0.01. **The p-value changed because n changed, not because the effect changed.**

For contrast, the length of the review differs a lot between the groups:

In [ ]:
compare_groups(reviews, "n_words").round(4)

### 2.2 Categorical outcome: label × verified purchase

A contingency table, the chi-square test of independence and Cramér's V.

In [ ]:
table = pd.crosstab(reviews["label"], reviews["verified_purchase"])
res = stats.chi2_contingency(table)
print(f"chi2 = {res.statistic:.2f}, dof = {res.dof}, p = {res.pvalue:.4f}")
print(f"Cramér's V = {association(table, method='cramer'):.3f}")
expected = pd.DataFrame(res.expected_freq, index=table.index, columns=table.columns).round(0)
display(table, expected)
pd.crosstab(reviews["label"], reviews["verified_purchase"], normalize="columns").round(3)

**Significant but negligible.** p ≈ 0.014 is below 0.05, but Cramér's V ≈ 0.013 and the share of negative reviews differs by only 1.5 percentage points (19.4 % vs 17.9 %).

**Exercise.** Write two sentences for a product manager: what was tested, what was found, and why the result should not change any decision. Then repeat the chi-square test on `full`: how do p and V change?

In [ ]:
# Exercise: chi-square on the full training set
# table_full = pd.crosstab(full["label"], full["verified_purchase"])
# ...

## Block 3 · Text length and helpful votes

### 3.1 Scatter plot and correlation

Both variables are right-skewed counts with many zeros, so we look at them on a log(1 + x) scale and compare Pearson's r (linear) with Spearman's ρ (monotonic, rank-based).

In [ ]:
words, votes = reviews["n_words"], reviews["helpful_vote"]
lw, lv = np.log1p(words), np.log1p(votes)

result = pd.Series({
    "Pearson r (raw)": stats.pearsonr(words, votes).statistic,
    "Spearman rho (raw)": stats.spearmanr(words, votes).statistic,
    "Pearson r (log scale)": stats.pearsonr(lw, lv).statistic,
})
print(result.round(3))

# sensitivity: drop the five reviews with the most votes
keep = ~votes.index.isin(votes.nlargest(5).index)
print("Pearson r without top 5:", round(stats.pearsonr(words[keep], votes[keep]).statistic, 3))

In [ ]:
sub = reviews.sample(5000, random_state=1)
rng = np.random.default_rng(0)
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
ax.scatter(np.log1p(sub["n_words"]), np.log1p(sub["helpful_vote"]) + rng.uniform(-0.08, 0.08, len(sub)),
           s=4, alpha=0.25, color="0.5", label="review (jittered)")
bins = pd.cut(lw, np.arange(0, 8.5, 0.5))
means = pd.DataFrame({"x": lw, "y": lv}).groupby(bins, observed=True).mean()
ax.plot(means["x"], means["y"], "o", color="#0072B2", label="mean per length bin")
b1, b0 = np.polyfit(lw, lv, 1)
grid = np.linspace(lw.min(), lw.max(), 2)
ax.plot(grid, b0 + b1 * grid, color="#D55E00", lw=2, label=f"least-squares line, slope {b1:.2f}")
ax.set(xlabel="log(1 + words)", ylabel="log(1 + helpful votes)",
       title="Longer reviews receive more helpful votes")
ax.legend(frameon=False);

### 3.2 Is review age a confounder?

Older reviews had more time to collect votes, and review length may have changed over the years. If both are true, part of the correlation could be due to age. A simple check: compute the correlation **within** each year.

In [ ]:
within = (reviews.query("year >= 2012").groupby("year")
          .apply(lambda g: pd.Series({"n": len(g),
                                      "spearman": stats.spearmanr(g["n_words"], g["helpful_vote"]).statistic,
                                      "median words": g["n_words"].median(),
                                      "mean votes": g["helpful_vote"].mean()}), include_groups=False))
within.round(3)

**Exercise.** Does the relationship hold within every year? Is it stronger or weaker than the pooled ρ? What does the trend in "mean votes" by year tell you about review age?

### 3.3 Simpson-type reversal: verified purchases by year

Pooled, verified purchases rate slightly lower. Within most years they rate higher, because unverified reviews were concentrated in 2015–2016, when ratings were high overall.

In [ ]:
full["year"] = full["date"].dt.year
by_year = full.query("year >= 2012").groupby(["year", "verified_purchase"])["rating"].mean().unstack()
by_year["verified minus not"] = by_year[True] - by_year[False]
by_year["unverified reviews"] = full.query("year >= 2012").groupby("year")["verified_purchase"].apply(lambda s: (~s).sum())
print("pooled difference:", round(full.groupby("verified_purchase")["rating"].mean().diff().iloc[-1], 3))
by_year.round(3)

### 3.4 A one-page summary for a product manager

Use the four-part structure for each finding: **finding** (one sentence), **number** with context, **meaning** for the decision, **limitation**.

**Exercise.** Write three findings below (ratings over time, verified purchases, length and helpful votes), each with one chart from this notebook. Alternatively, extend the Streamlit dashboard in `dashboard_app.py` (start it from the repository root with `uv run streamlit run sessions/05-eda-and-statistics/workbooks/dashboard_app.py`).

*Your summary here.*

1. **Finding:** …  **Number:** …  **Meaning:** …  **Limitation:** …
2. …
3. …